In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

In [2]:
# ## 1. Подготовка табличного датасета и утилит

class TabularDataset(Dataset):
    def __init__(self, X: pd.DataFrame, y: pd.Series, cat_cols, num_cols, scaler: StandardScaler=None):
        self.cat_cols = cat_cols
        self.num_cols = num_cols
        # числовые признаки
        if scaler is None:
            self.scaler = StandardScaler()
            self.numeric = self.scaler.fit_transform(X[num_cols])
        else:
            self.scaler = scaler
            self.numeric = self.scaler.transform(X[num_cols])
        # категориальные: кодируем категории в числа [0..N-1], неизвестные => -1
        codes = np.stack([
            X[c].astype("category").cat.codes.values for c in cat_cols
        ], axis=1)
        # все -1 => 0, чтобы не было out-of-bounds в Embedding
        codes[codes < 0] = 0
        self.cat_codes = codes
        self.y = y.values.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "numeric": torch.tensor(self.numeric[idx], dtype=torch.float32),
            "categorical": torch.tensor(self.cat_codes[idx], dtype=torch.long),
            "target": torch.tensor(self.y[idx], dtype=torch.long)
        }


def load_and_split(path, test_size=0.2, random_state=42):
    df = pd.read_csv(path)
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    X = df.drop(columns=['radiant_win'])
    y = df['radiant_win']
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
    return X_train, X_val, y_train, y_val, num_cols, cat_cols

# Путь к данным
DATA_PATH = '/kaggle/input/daotka/result.csv'
X_train, X_val, y_train, y_val, num_cols, cat_cols = load_and_split(DATA_PATH)

In [3]:
# ## 2. MLP с эмбеддингами + BatchNorm + Dropout


class MLPEmbeddingModel(nn.Module):
    def __init__(self, cat_dims, emb_dims, num_inp, hidden_dims=[200,100], dropout=0.5):
        super().__init__()
        self.embs = nn.ModuleList([
            nn.Embedding(cat_dims[i], emb_dims[i]) for i in range(len(cat_dims))
        ])
        self.bn_num = nn.BatchNorm1d(num_inp)
        inp_dim = sum(emb_dims) + num_inp
        layers = []
        for h in hidden_dims:
            layers += [
                nn.Linear(inp_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            inp_dim = h
        layers.append(nn.Linear(inp_dim, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        emb = [self.embs[i](x_cat[:,i]) for i in range(x_cat.size(1))]
        x = torch.cat([*emb, self.bn_num(x_num)], dim=1)
        return self.net(x)


# Параметры
cat_dims = [int(X_train[c].nunique()) for c in cat_cols]
emb_dims = [min(50, (cd+1)//2) for cd in cat_dims]

# Датасеты и загрузчики
train_ds = TabularDataset(X_train, y_train, cat_cols, num_cols)
val_ds   = TabularDataset(X_val,   y_val,   cat_cols, num_cols, scaler=train_ds.scaler)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256)

# Модель, оптимизатор, лосс
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLPEmbeddingModel(cat_dims, emb_dims, len(num_cols), hidden_dims=[200,100], dropout=0.5).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

/usr/local/lib/python3.11/dist-packages/sklearn/utils/extmath.py:1047: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.11/dist-packages/sklearn/utils/extmath.py:1052: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.11/dist-packages/sklearn/utils/extmath.py:1072: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_data.py:87: RuntimeWarning: invalid value encountered in less_equal
  return var <= upper_bound


In [4]:
# Тренировка
n_epochs = 20
history = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

for epoch in range(n_epochs):
    model.train()
    total_loss, correct = 0, 0
    for b in train_loader:
        x_num = b["numeric"].to(device)
        x_cat = b["categorical"].to(device)
        y = b["target"].to(device)
        opt.zero_grad()
        logits = model(x_num, x_cat)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()*y.size(0)
        correct += (logits.argmax(1)==y).sum().item()
    train_loss = total_loss/len(train_ds)
    train_acc  = correct/len(train_ds)

    # eval
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for b in val_loader:
            x_num = b["numeric"].to(device)
            x_cat = b["categorical"].to(device)
            y = b["target"].to(device)
            logits = model(x_num, x_cat)
            loss = criterion(logits, y)
            total_loss += loss.item()*y.size(0)
            correct += (logits.argmax(1)==y).sum().item()
    val_loss = total_loss/len(val_ds)
    val_acc  = correct/len(val_ds)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{n_epochs}  "
          f"train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

Epoch 1/20  train_acc=0.4976 val_acc=0.4977
Epoch 2/20  train_acc=0.4976 val_acc=0.4977
Epoch 3/20  train_acc=0.4976 val_acc=0.4977
Epoch 4/20  train_acc=0.4976 val_acc=0.4977
Epoch 5/20  train_acc=0.4976 val_acc=0.4977
Epoch 6/20  train_acc=0.4976 val_acc=0.4977
Epoch 7/20  train_acc=0.4976 val_acc=0.4977
Epoch 8/20  train_acc=0.4976 val_acc=0.4977
Epoch 9/20  train_acc=0.4976 val_acc=0.4977
Epoch 10/20  train_acc=0.4976 val_acc=0.4977
Epoch 11/20  train_acc=0.4976 val_acc=0.4977
Epoch 12/20  train_acc=0.4976 val_acc=0.4977
Epoch 13/20  train_acc=0.4976 val_acc=0.4977
Epoch 14/20  train_acc=0.4976 val_acc=0.4977
Epoch 15/20  train_acc=0.4976 val_acc=0.4977
Epoch 16/20  train_acc=0.4976 val_acc=0.4977
Epoch 17/20  train_acc=0.4976 val_acc=0.4977
Epoch 18/20  train_acc=0.4976 val_acc=0.4977
Epoch 19/20  train_acc=0.4976 val_acc=0.4977
Epoch 20/20  train_acc=0.4976 val_acc=0.4977


In [6]:
# ## 3. TabTransformer (Self-Attention)

# %% [code]
class TabTransformerModel(nn.Module):
    def __init__(self, cat_dims, emb_dim, num_inp, n_heads=8, n_layers=2, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(cat_dims[i], emb_dim) for i in range(len(cat_dims))])
        self.num_proj = nn.Linear(num_inp, emb_dim)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=n_heads, dropout=dropout)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear((len(cat_dims)+1)*emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )
    def forward(self, x_num, x_cat):
        cat_tok = torch.stack([self.embs[i](x_cat[:,i]) for i in range(x_cat.size(1))], dim=1)
        num_tok = self.num_proj(x_num).unsqueeze(1)
        tokens = torch.cat([cat_tok, num_tok], dim=1)  # [B, seq, emb_dim]
        # Transformer: expects [seq, B, emb_dim]
        out = self.transformer(tokens.permute(1,0,2))
        out = out.permute(1,0,2)
        return self.head(out)

# Параметры
emb_dim = 32
model_tt = TabTransformerModel(cat_dims, emb_dim, len(num_cols), n_heads=4, n_layers=2, hidden_dim=100, dropout=0.3).to(device)
opt_tt = torch.optim.AdamW(model_tt.parameters(), lr=5e-4)
crit_tt = nn.CrossEntropyLoss()

# Тренировка TabTransformer
n_epochs = 20
hist_tt = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [7]:
for epoch in range(n_epochs):
    # train
    model_tt.train()
    total_loss, correct = 0, 0
    for b in train_loader:
        x_num = b["numeric"].to(device)
        x_cat = b["categorical"].to(device)
        y = b["target"].to(device)
        opt_tt.zero_grad()
        logits = model_tt(x_num, x_cat)
        loss = crit_tt(logits, y)
        loss.backward()
        opt_tt.step()
        total_loss += loss.item()*y.size(0)
        correct += (logits.argmax(1)==y).sum().item()
    tr_loss = total_loss/len(train_ds)
    tr_acc  = correct/len(train_ds)
    # eval
    model_tt.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for b in val_loader:
            x_num = b["numeric"].to(device)
            x_cat = b["categorical"].to(device)
            y = b["target"].to(device)
            logits = model_tt(x_num, x_cat)
            loss = crit_tt(logits, y)
            total_loss += loss.item()*y.size(0)
            correct += (logits.argmax(1)==y).sum().item()
    va_loss = total_loss/len(val_ds)
    va_acc  = correct/len(val_ds)

    hist_tt["train_loss"].append(tr_loss)
    hist_tt["train_acc"].append(tr_acc)
    hist_tt["val_loss"].append(va_loss)
    hist_tt["val_acc"].append(va_acc)

    print(f"TT Epoch {epoch+1}/{n_epochs}  train_acc={tr_acc:.4f} val_acc={va_acc:.4f}")

TT Epoch 1/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 2/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 3/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 4/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 5/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 6/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 7/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 8/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 9/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 10/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 11/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 12/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 13/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 14/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 15/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 16/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 17/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 18/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 19/20  train_acc=0.4976 val_acc=0.4977
TT Epoch 20/20  train_acc=0.4976 val_acc=0.4977


In [12]:
from sklearn.impute import SimpleImputer

class TabularDataset(Dataset):
    def __init__(self, X: pd.DataFrame, y: pd.Series, cat_cols, num_cols, scaler: StandardScaler=None):
        self.cat_cols = cat_cols
        self.num_cols = num_cols

        # 1) Заполняем пропуски по среднему
        num_data = X[num_cols].copy()
        imp = SimpleImputer(strategy="mean")
        num_filled = imp.fit_transform(num_data)

        # 2) Стандартизация
        if scaler is None:
            self.scaler = StandardScaler()
            self.numeric = self.scaler.fit_transform(num_filled)
        else:
            self.scaler = scaler
            self.numeric = self.scaler.transform(num_filled)

        # 3) Категории → коды [0..N-1], все –1 → 0
        codes = np.stack([
            X[c].astype("category").cat.codes.values for c in cat_cols
        ], axis=1)
        codes[codes < 0] = 0
        self.cat_codes = codes

        # 4) Целевая
        self.y = y.values.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "numeric": torch.tensor(self.numeric[idx], dtype=torch.float32),
            "categorical": torch.tensor(self.cat_codes[idx], dtype=torch.long),
            "target": torch.tensor(self.y[idx], dtype=torch.long)
        }

In [15]:
# 4. TabNet

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from pytorch_tabnet.tab_model import TabNetClassifier

# 1) Иммутация и масштабирование числовых признаков
imp = SimpleImputer(strategy="mean")
scaler = StandardScaler()

num_train = imp.fit_transform(X_train[num_cols])
num_val   = imp.transform(X_val[num_cols])

num_train = scaler.fit_transform(num_train)
num_val   = scaler.transform(num_val)

# 2) Кодирование категорий и замена -1 → 0
cat_train = np.stack([
    X_train[c].astype("category").cat.codes.replace(-1, 0).values 
    for c in cat_cols
], axis=1)
cat_val = np.stack([
    X_val[c].astype("category").cat.codes.replace(-1, 0).values 
    for c in cat_cols
], axis=1)

# 3) Собираем финальные numpy-массивы
train_array = np.concatenate([num_train, cat_train], axis=1)
val_array   = np.concatenate([num_val,   cat_val],   axis=1)
y_train_arr = y_train.values
y_val_arr   = y_val.values

# 4) Параметры и обучение TabNet
num_feats = num_train.shape[1]
cat_idxs = list(range(num_feats, num_feats + len(cat_cols)))
cat_dims = [int(X_train[c].nunique()) for c in cat_cols]

clf = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=8,
    n_d=16, n_a=16, n_steps=5, gamma=1.5,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={"lr": 2e-3},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size": 10, "gamma": 0.9},
    mask_type="sparsemax"
)

clf.fit(
    train_array, y_train_arr,
    eval_set=[(train_array, y_train_arr), (val_array, y_val_arr)],
    eval_name=["train", "valid"],
    eval_metric=["accuracy"],
    max_epochs=50,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False
)

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.90928 | train_accuracy: 0.57689 | valid_accuracy: 0.57266 |  0:00:11s
epoch 1  | loss: 0.70274 | train_accuracy: 0.71686 | valid_accuracy: 0.71134 |  0:00:23s
epoch 2  | loss: 0.57727 | train_accuracy: 0.7762  | valid_accuracy: 0.77915 |  0:00:34s
epoch 3  | loss: 0.51118 | train_accuracy: 0.80283 | valid_accuracy: 0.80242 |  0:00:45s
epoch 4  | loss: 0.47339 | train_accuracy: 0.81998 | valid_accuracy: 0.81558 |  0:00:57s
epoch 5  | loss: 0.43574 | train_accuracy: 0.83604 | valid_accuracy: 0.83885 |  0:01:08s
epoch 6  | loss: 0.34288 | train_accuracy: 0.94162 | valid_accuracy: 0.94349 |  0:01:19s
epoch 7  | loss: 0.18587 | train_accuracy: 0.95964 | valid_accuracy: 0.95865 |  0:01:30s
epoch 8  | loss: 0.14478 | train_accuracy: 0.96941 | valid_accuracy: 0.96583 |  0:01:41s
epoch 9  | loss: 0.12608 | train_accuracy: 0.97257 | valid_accuracy: 0.97288 |  0:01:53s
epoch 10 | loss: 0.10241 | train_accuracy: 0.97979 | valid_accuracy: 0.97966 |  0:02:04s
epoch 11 | loss: 0.08

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
